# MCI to AD Converter Visit Extraction

This notebook derives converter and non-converter longitudinal trajectories from first-visit MCI and AD cohorts.

## Criteria Table

| Cohort/Group | Source Definition | Inclusion Rule | Decision |
|---|---|---|---|
| MCI source cohort | From `MCI_first_visit.csv` | Patient appears in MCI first-visit file | Candidate for converter analysis |
| AD source cohort | From `AD_use_only_cognitive_scores_first_visit.csv` | Patient appears in AD first-visit file | Used to label converters |
| Converter | Intersection of MCI and AD source cohorts | Patient in both source cohorts | Keep visits from first MCI visit onward |
| Non-converter | MCI cohort minus AD cohort | Patient in MCI source and not in AD source | Keep visits from first MCI visit onward |

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Centralized paths ──────────────────────────────────────────────────
BASE = Path('/dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION')
INTERMEDIATE = BASE / 'data' / 'intermediate'

mci_first_visit_csv = INTERMEDIATE / 'mci' / 'first_visit.csv'
ad_first_visit_csv  = INTERMEDIATE / 'ad'  / 'cognitive_first_visit.csv'
full_input_csv      = INTERMEDIATE / 'combined' / 'all_visits.csv'

converter_dir     = INTERMEDIATE / 'mci' / 'converter'
non_converter_dir = INTERMEDIATE / 'mci' / 'non_converter'
converter_dir.mkdir(parents=True, exist_ok=True)
non_converter_dir.mkdir(parents=True, exist_ok=True)

conv_xlsx    = converter_dir / 'highlighted.xlsx'
conv_csv     = converter_dir / 'all_visits.csv'
nonconv_xlsx = non_converter_dir / 'highlighted.xlsx'
nonconv_csv  = non_converter_dir / 'all_visits.csv'

mci = pd.read_csv(mci_first_visit_csv)
ad = pd.read_csv(ad_first_visit_csv)
full = pd.read_csv(full_input_csv)

# Convert numeric columns
for df in [mci, ad, full]:
    df['mmstot'] = pd.to_numeric(df['mmstot'], errors='coerce')
    df['cdrglobal'] = pd.to_numeric(df['cdrglobal'], errors='coerce')

print(f'Input MCI first-visit CSV: {mci_first_visit_csv}')
print(f'Input AD first-visit CSV:  {ad_first_visit_csv}')
print(f'Input full CSV:            {full_input_csv}')
print(f'Converter output dir:      {converter_dir}')
print(f'Non-converter output dir:  {non_converter_dir}')
print(f'MCI first visits: {len(mci)} patients')
print(f'AD first visits:  {len(ad)} patients')
print(f'Full dataset:     {len(full)} rows')
non_empty = (full["visdate"] != "").sum()
print(f'visdate non-empty: {non_empty}, empty: {len(full) - non_empty}')
print(f'Sample visdate (date-only):')
print(full["visdate"].head(3).tolist())


Input MCI first-visit CSV: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/first_visit.csv
Input AD first-visit CSV:  /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/ad/cognitive_first_visit.csv
Input full CSV:            /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/combined/all_visits.csv
Converter output dir:      /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/converter
Non-converter output dir:  /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/non_converter
MCI first visits: 490 patients
AD first visits:  108 patients
Full dataset:     4605 rows
visdate non-empty: 4605, empty: 0
Sample visdate (date-only):
['01-10-2015', '12-10-2016', '20-09-2017']


## Combined MCI + AD Overview

Show all visits for patients in the MCI or AD cohorts, with highlighted rows:
- **Orange**: MCI visits (from MCI all-visits dataset)
- **Dark red**: Strict AD visits (cognitive + biomarkers)
- **Light red**: Cognitive-only AD visits

In [12]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

# ── Load MCI and AD all-visits to identify highlighted rows ──
mci_all = pd.read_csv(INTERMEDIATE / "mci" / "all_visits.csv")
ad_strict_all = pd.read_csv(INTERMEDIATE / "ad" / "strict_all_visits.csv")
ad_cog_all = pd.read_csv(INTERMEDIATE / "ad" / "cognitive_all_visits.csv")

# Build visit keys for highlighting (Repseudonym, visdate)
mci_visit_keys = set(zip(mci_all["Repseudonym"], mci_all["visdate"]))
strict_ad_visit_keys = set(zip(ad_strict_all["Repseudonym"], ad_strict_all["visdate"]))
cog_ad_visit_keys = set(zip(ad_cog_all["Repseudonym"], ad_cog_all["visdate"])) - strict_ad_visit_keys

# Patients in either MCI or AD
mci_patient_ids = set(mci["Repseudonym"])
ad_patient_ids = set(ad["Repseudonym"])
combined_ids = mci_patient_ids | ad_patient_ids

# Get all visits for combined patients
df_combined = full[full["Repseudonym"].isin(combined_ids)].copy()

# Classify each row
orange_indices = set()
dark_red_indices = set()
light_red_indices = set()

for idx, row in df_combined.iterrows():
    key = (row["Repseudonym"], row["visdate"])
    if key in strict_ad_visit_keys:
        dark_red_indices.add(idx)
    elif key in cog_ad_visit_keys:
        light_red_indices.add(idx)
    elif key in mci_visit_keys:
        orange_indices.add(idx)

print(f"Combined MCI + AD patients: {len(combined_ids)}")
print(f"Total visits: {len(df_combined)}")
print(f"  Orange (MCI): {len(orange_indices)}")
print(f"  Dark red (strict AD): {len(dark_red_indices)}")
print(f"  Light red (cognitive-only AD): {len(light_red_indices)}")
print(f"  Uncolored: {len(df_combined) - len(orange_indices) - len(dark_red_indices) - len(light_red_indices)}")

# ── Write combined highlighted Excel ──
combined_xlsx = INTERMEDIATE / "mci" / "combined_mci_ad_highlighted.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "MCI + AD Visits"

display_cols = ["file", "Repseudonym", "visdate", "scan_date", "scan_year", "prmdiag",
                "mmstot", "cdrglobal", "ratio_Abeta42_40", "ratio_Abeta42_phosphotau181"]
display_cols = [c for c in display_cols if c in df_combined.columns]

orange_fill = PatternFill(start_color="FBE7CF", end_color="FBE7CF", fill_type="solid")
dark_red_fill = PatternFill(start_color="ff0606", end_color="ff0606", fill_type="solid")
light_red_fill = PatternFill(start_color="ff8080", end_color="ff8080", fill_type="solid")
white_font = Font(color="FFFFFF")
header_fill = PatternFill(start_color="333333", end_color="333333", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)

# Header
for col_idx, col_name in enumerate(display_cols, 1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

# Data rows
for excel_row, (df_idx, row_data) in enumerate(df_combined.iterrows(), 2):
    is_dark_red = df_idx in dark_red_indices
    is_light_red = df_idx in light_red_indices
    is_orange = df_idx in orange_indices
    for col_idx, col_name in enumerate(display_cols, 1):
        value = row_data[col_name]
        if pd.isna(value):
            value = ""
        cell = ws.cell(row=excel_row, column=col_idx, value=str(value))
        if is_dark_red:
            cell.fill = dark_red_fill
            cell.font = white_font
        elif is_light_red:
            cell.fill = light_red_fill
        elif is_orange:
            cell.fill = orange_fill

# Auto-adjust column widths
for col_idx, col_name in enumerate(display_cols, 1):
    max_len = len(str(col_name))
    for row in ws.iter_rows(min_row=2, max_row=min(100, ws.max_row), min_col=col_idx, max_col=col_idx):
        for cell in row:
            if cell.value:
                max_len = max(max_len, len(str(cell.value)))
    ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = min(max_len + 2, 40)

wb.save(combined_xlsx)
print(f"\n✓ Combined highlighted Excel saved to: {combined_xlsx}")
print(f"  File size: {combined_xlsx.stat().st_size:,} bytes")

Combined MCI + AD patients: 559
Total visits: 2944
  Orange (MCI): 1304
  Dark red (strict AD): 33
  Light red (cognitive-only AD): 203
  Uncolored: 1404

✓ Combined highlighted Excel saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/combined_mci_ad_highlighted.xlsx
  File size: 148,620 bytes


In [13]:
# Identify converters and non-converters
mci_patients = set(mci['Repseudonym'])
ad_patients = set(ad['Repseudonym'])

converter_ids = mci_patients & ad_patients
non_converter_ids = mci_patients - ad_patients

print(f'Converters (MCI → AD):  {len(converter_ids)}')
print(f'Non-converters (MCI only): {len(non_converter_ids)}')
print(f'AD-only (not in MCI):      {len(ad_patients - mci_patients)} (not used)')

Converters (MCI → AD):  39
Non-converters (MCI only): 451
AD-only (not in MCI):      69 (not used)


In [14]:
def find_matching_idx(patient_visits, first_visit_row):
    """Find the index in the full dataset that matches a first-visit row.
    Matches by mmstot + cdrglobal + file + visnam.
    Falls back to just mmstot + cdrglobal if no exact match."""
    fv = first_visit_row
    
    # Try exact match on all fields
    mask = (
        (patient_visits['mmstot'] == fv['mmstot']) &
        (patient_visits['cdrglobal'] == fv['cdrglobal']) &
        (patient_visits['file'] == fv['file'])
    )
    # Also match visnam if available
    if pd.notna(fv.get('visnam')):
        mask = mask & (patient_visits['visnam'] == fv['visnam'])
    
    matches = patient_visits[mask]
    if len(matches) > 0:
        return matches.index[0]
    
    # Fallback: match on mmstot + cdrglobal only
    mask_fb = (
        (patient_visits['mmstot'] == fv['mmstot']) &
        (patient_visits['cdrglobal'] == fv['cdrglobal'])
    )
    matches_fb = patient_visits[mask_fb]
    if len(matches_fb) > 0:
        return matches_fb.index[0]
    
    # Last resort: return first visit index
    return patient_visits.index[0]

print('Matching function defined.')

Matching function defined.


In [15]:
# Extract converter trajectories
# Also load strict AD first-visit CSV to determine ad_criteria
ad_strict_first = pd.read_csv(INTERMEDIATE / 'ad' / 'strict_first_visit.csv')
strict_ad_patients = set(ad_strict_first['Repseudonym'])

# Load strict AD all-visits to identify which visits meet strict criteria
ad_strict_all = pd.read_csv(INTERMEDIATE / 'ad' / 'strict_all_visits.csv')
strict_ad_visit_keys = set(zip(ad_strict_all['Repseudonym'], ad_strict_all['visdate']))

# Load cognitive-only AD all-visits
ad_cog_all = pd.read_csv(INTERMEDIATE / 'ad' / 'cognitive_all_visits.csv')
cog_ad_visit_keys = set(zip(ad_cog_all['Repseudonym'], ad_cog_all['visdate']))

converter_rows = []
converter_ad_indices = set()       # strict AD (dark red)
converter_ad_cog_indices = set()   # cognitive-only AD (light red)
converter_mci_indices = set()      # MCI phase (orange)

match_issues = []

for pid in sorted(converter_ids):
    patient_visits = full[full['Repseudonym'] == pid].copy()
    mci_fv = mci[mci['Repseudonym'] == pid].iloc[0]
    ad_fv = ad[ad['Repseudonym'] == pid].iloc[0]

    mci_idx = find_matching_idx(patient_visits, mci_fv)
    ad_idx = find_matching_idx(patient_visits, ad_fv)

    if mci_idx > ad_idx:
        match_issues.append(f'{pid}: MCI idx {mci_idx} > AD idx {ad_idx}')

    visits_from_mci = patient_visits.loc[mci_idx:]

    for idx in visits_from_mci.index:
        converter_rows.append(idx)
        row = patient_visits.loc[idx]
        visit_key = (pid, row['visdate'])

        if visit_key in strict_ad_visit_keys:
            converter_ad_indices.add(idx)      # strict AD (dark red)
        elif visit_key in cog_ad_visit_keys:
            converter_ad_cog_indices.add(idx)  # cognitive-only AD (light red)
        elif idx >= ad_idx:
            converter_ad_cog_indices.add(idx)  # after AD onset
        else:
            converter_mci_indices.add(idx)     # MCI phase (orange)

df_converters = full.loc[converter_rows].copy()

# Add ad_criteria column
df_converters['ad_criteria'] = ''
for idx in converter_ad_indices:
    df_converters.at[idx, 'ad_criteria'] = 'STRICT'
for idx in converter_ad_cog_indices:
    if df_converters.at[idx, 'ad_criteria'] == '':
        df_converters.at[idx, 'ad_criteria'] = 'COGNITIVE_ONLY'

print(f'Converter trajectories extracted:')
print(f'  Patients: {len(converter_ids)}')
print(f'  Total visits: {len(df_converters)}')
print(f'  MCI visits (orange): {len(converter_mci_indices)}')
print(f'  AD visits - strict (dark red): {len(converter_ad_indices)}')
print(f'  AD visits - cognitive only (light red): {len(converter_ad_cog_indices)}')

if match_issues:
    print(f'\n⚠ Match issues ({len(match_issues)}):')
    for issue in match_issues:
        print(f'  {issue}')
else:
    print(f'\n✓ No match issues — MCI always before AD for all converters')


Converter trajectories extracted:
  Patients: 39
  Total visits: 191
  MCI visits (orange): 113
  AD visits - strict (dark red): 5
  AD visits - cognitive only (light red): 73

⚠ Match issues (1):
  8de59cd14: MCI idx 2601 > AD idx 2600


## Compute Conversion Time
For converters: `conversion_time_from_mci_to_ad_days` = `first_ad_visit` − `first_mci_visit` (days)
and `conversion_time_from_mci_to_ad_years` = days / 365.25.

In [16]:
# ── Compute conversion time for converters ──────────────────────────────

# Build first_mci_visit and first_ad_visit lookup from the source CSVs
# (these now contain scan_date and first_mci_visit / first_ad_visit columns)

# first_mci_visit from the MCI first-visit CSV
if 'first_mci_visit' in mci.columns:
    mci_lookup = mci.set_index('Repseudonym')['first_mci_visit']
elif 'scan_date' in mci.columns:
    mci_lookup = mci.set_index('Repseudonym')['scan_date']
else:
    mci_lookup = pd.Series(dtype=str)

# first_ad_visit from the AD first-visit CSV
if 'first_ad_visit' in ad.columns:
    ad_lookup = ad.set_index('Repseudonym')['first_ad_visit']
elif 'scan_date' in ad.columns:
    ad_lookup = ad.set_index('Repseudonym')['scan_date']
else:
    ad_lookup = pd.Series(dtype=str)

# Map to converter DataFrame
df_converters['first_mci_visit'] = df_converters['Repseudonym'].map(mci_lookup)
df_converters['first_ad_visit'] = df_converters['Repseudonym'].map(ad_lookup)

# Parse dates and compute conversion time
mci_dt = pd.to_datetime(df_converters['first_mci_visit'], errors='coerce')
ad_dt = pd.to_datetime(df_converters['first_ad_visit'], errors='coerce')

df_converters['conversion_time_from_mci_to_ad_days'] = (ad_dt - mci_dt).dt.days
df_converters['conversion_time_from_mci_to_ad_years'] = (df_converters['conversion_time_from_mci_to_ad_days'] / 365.25).round(2)

n_computed = df_converters['conversion_time_from_mci_to_ad_days'].notna().sum()
print(f'Conversion time computed for {n_computed} / {len(df_converters)} converter rows')
print(f'Unique patients with conversion time: {df_converters.dropna(subset=["conversion_time_from_mci_to_ad_days"])["Repseudonym"].nunique()}')

# Show summary
conv_summary = (df_converters.dropna(subset=['conversion_time_from_mci_to_ad_days'])
                .drop_duplicates('Repseudonym')
                [['Repseudonym', 'first_mci_visit', 'first_ad_visit', 'conversion_time_from_mci_to_ad_days', 'conversion_time_from_mci_to_ad_years']])
print(f'\nConversion time statistics:')
print(f'  Mean: {conv_summary["conversion_time_from_mci_to_ad_years"].mean():.1f} years')
print(f'  Median: {conv_summary["conversion_time_from_mci_to_ad_years"].median():.1f} years')
print(f'  Range: {conv_summary["conversion_time_from_mci_to_ad_years"].min():.1f} – {conv_summary["conversion_time_from_mci_to_ad_years"].max():.1f} years')
conv_summary.head(10)

Conversion time computed for 191 / 191 converter rows
Unique patients with conversion time: 39

Conversion time statistics:
  Mean: 3.2 years
  Median: 3.0 years
  Range: -1.1 – 8.0 years


/dss/lxclscratch/0A/di54lup/di54lup/ipykernel_138563/1059796761.py:27: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  mci_dt = pd.to_datetime(df_converters['first_mci_visit'], errors='coerce')
/dss/lxclscratch/0A/di54lup/di54lup/ipykernel_138563/1059796761.py:28: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  ad_dt = pd.to_datetime(df_converters['first_ad_visit'], errors='coerce')


,Repseudonym,first_mci_visit,first_ad_visit,conversion_time_from_mci_to_ad_days,conversion_time_from_mci_to_ad_years
48,03a0a6663,29-10-2015,22-10-2018,1089,2.98
57,040356085,17-03-2017,03-04-2019,747,2.05
92,0588be6c5,15-12-2017,04-05-2021,1236,3.38
181,0df733308,10-12-2014,28-11-2017,1084,2.97
327,14532a273,07-05-2015,18-06-2018,1138,3.12
413,191e8e85d,26-01-2018,14-01-2020,718,1.97
611,220e18c57,14-06-2017,13-06-2018,364,1.00
855,2d73e8996,03-02-2015,07-12-2022,2864,7.84
865,2dd25390a,10-01-2018,21-12-2018,345,0.94
869,2e2857631,25-09-2017,19-11-2020,1151,3.15


In [17]:
# Extract non-converter trajectories
non_converter_rows = []
non_converter_mci_indices = set()

for pid in sorted(non_converter_ids):
    patient_visits = full[full['Repseudonym'] == pid].copy()
    mci_fv = mci[mci['Repseudonym'] == pid].iloc[0]
    mci_idx = find_matching_idx(patient_visits, mci_fv)

    visits_from_mci = patient_visits.loc[mci_idx:]
    for idx in visits_from_mci.index:
        non_converter_rows.append(idx)
        non_converter_mci_indices.add(idx)

df_non_converters = full.loc[non_converter_rows].copy()

# Add first_mci_visit from the MCI first-visit CSV
if 'first_mci_visit' in mci.columns:
    mci_lookup = mci.set_index('Repseudonym')['first_mci_visit']
else:
    # Fallback: compute from scan_date / visdate
    tmp = mci.copy()
    tmp['_date'] = pd.to_datetime(tmp.get('scan_date', pd.Series(dtype=str)), dayfirst=True, errors='coerce')
    tmp['_date'] = tmp['_date'].fillna(pd.to_datetime(tmp['visdate'], dayfirst=True, errors='coerce'))
    mci_lookup = tmp.set_index('Repseudonym')['_date'].dt.strftime('%d-%m-%Y')

df_non_converters['first_mci_visit'] = df_non_converters['Repseudonym'].map(mci_lookup)

print(f'Non-converter trajectories extracted:')
print(f'  Patients: {len(non_converter_ids)}')
print(f'  Total visits: {len(df_non_converters)}')
print(f'  first_mci_visit populated: {df_non_converters["first_mci_visit"].notna().sum()} / {len(df_non_converters)}')


Non-converter trajectories extracted:
  Patients: 451
  Total visits: 2113
  first_mci_visit populated: 2113 / 2113


In [18]:
# Summary
print('=' * 70)
print('CONVERTER VISIT EXTRACTION SUMMARY')
print('=' * 70)
print(f'\nConverters (MCI → AD): {len(converter_ids)} patients, {len(df_converters)} visits')
print(f'  MCI phase visits (orange): {len(converter_mci_indices)}')
print(f'  AD phase visits (red):     {len(converter_ad_indices)}')
print(f'\nNon-converters: {len(non_converter_ids)} patients, {len(df_non_converters)} visits')
print(f'  All visits colored orange')

# Avg visits per group
avg_conv = len(df_converters) / max(len(converter_ids), 1)
avg_nonconv = len(df_non_converters) / max(len(non_converter_ids), 1)
print(f'\nAvg visits per converter: {avg_conv:.1f}')
print(f'Avg visits per non-converter: {avg_nonconv:.1f}')

CONVERTER VISIT EXTRACTION SUMMARY

Converters (MCI → AD): 39 patients, 191 visits
  MCI phase visits (orange): 113
  AD phase visits (red):     5

Non-converters: 451 patients, 2113 visits
  All visits colored orange

Avg visits per converter: 4.9
Avg visits per non-converter: 4.7


In [19]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font

def write_highlighted_excel(df, orange_indices, red_indices, light_red_indices, output_path, sheet_name):
    wb = Workbook()
    ws = wb.active
    ws.title = sheet_name

    cols = ['file','Repseudonym','visdate','scan_date','scan_year','prmdiag','mmstot','cdrglobal',
            'ratio_Abeta42_40','ratio_Abeta42_phosphotau181',
            'first_mci_visit','first_ad_visit','conversion_time_from_mci_to_ad_days','conversion_time_from_mci_to_ad_years',
            'ad_criteria']
    cols = [c for c in cols if c in df.columns]

    # Styles
    orange_fill = PatternFill(start_color='FBE7CF', end_color='FBE7CF', fill_type='solid')
    dark_red_fill = PatternFill(start_color='ff0606', end_color='ff0606', fill_type='solid')
    light_red_fill = PatternFill(start_color='ff8080', end_color='ff8080', fill_type='solid')
    white_font = Font(color='FFFFFF')
    header_fill = PatternFill(start_color='333333', end_color='333333', fill_type='solid')
    header_font = Font(color='FFFFFF', bold=True)

    # Header
    for col_idx, col_name in enumerate(cols, 1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font

    # Data with highlighting
    for excel_row, (df_idx, row_data) in enumerate(df.iterrows(), 2):
        is_dark_red = df_idx in red_indices
        is_light_red = df_idx in light_red_indices
        is_orange = df_idx in orange_indices
        for col_idx, col_name in enumerate(cols, 1):
            value = row_data[col_name]
            if pd.isna(value):
                value = ""
            cell = ws.cell(row=excel_row, column=col_idx, value=str(value))
            if is_dark_red:
                cell.fill = dark_red_fill
                cell.font = white_font
            elif is_light_red:
                cell.fill = light_red_fill
            elif is_orange:
                cell.fill = orange_fill

    # Auto-adjust column widths
    for col_idx, col_name in enumerate(cols, 1):
        max_len = len(str(col_name))
        for row in ws.iter_rows(min_row=2, max_row=min(100, ws.max_row), min_col=col_idx, max_col=col_idx):
            for cell in row:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[ws.cell(row=1, column=col_idx).column_letter].width = min(max_len + 2, 40)

    wb.save(output_path)
    return output_path

# Write converter Excel — 3 colors: orange (MCI), dark red (strict AD), light red (cognitive-only AD)
write_highlighted_excel(df_converters, converter_mci_indices, converter_ad_indices, converter_ad_cog_indices, conv_xlsx, "Converters")
df_converters.to_csv(conv_csv, index=False)

print(f"✓ Converter Excel saved to: {conv_xlsx}")
print(f"✓ Converter CSV   saved to: {conv_csv}")
print(f"  Rows: {len(df_converters)}")
print(f"  Orange (MCI): {len(converter_mci_indices)}")
print(f"  Dark red (strict AD): {len(converter_ad_indices)}")
print(f"  Light red (cognitive-only AD): {len(converter_ad_cog_indices)}")


✓ Converter Excel saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/converter/highlighted.xlsx
✓ Converter CSV   saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/converter/all_visits.csv
  Rows: 191
  Orange (MCI): 113
  Dark red (strict AD): 5
  Light red (cognitive-only AD): 73


In [20]:
# Write non-converter Excel + CSV
write_highlighted_excel(df_non_converters, non_converter_mci_indices, set(), set(), nonconv_xlsx, "Non-Converters")
df_non_converters.to_csv(nonconv_csv, index=False)

print(f"✓ Non-converter Excel saved to: {nonconv_xlsx}")
print(f"✓ Non-converter CSV   saved to: {nonconv_csv}")
print(f"  Rows: {len(df_non_converters)}")
print(f"  Orange (MCI): {len(non_converter_mci_indices)}")


✓ Non-converter Excel saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/non_converter/highlighted.xlsx
✓ Non-converter CSV   saved to: /dss/dsshome1/0A/di54lup/DELCODE/STRATIFICATION/data/intermediate/mci/non_converter/all_visits.csv
  Rows: 2113
  Orange (MCI): 2113
